# LDPC码：构造与译码

**Low-Density Parity-Check Codes**

本notebook系统介绍LDPC码的理论基础、构造方法、译码算法以及5G NR标准中的应用。

## 1. 学习目标

通过本notebook的学习，你将：

- 理解LDPC码的核心概念：校验矩阵H、生成矩阵G、Tanner图
- 掌握Gallius方法构造LDPC校验矩阵
- 实现Sum-Product（和积）消息传递译码算法
- 理解密度演化分析的基本原理
- 了解5G NR LDPC码的基矩阵（BG）和移位因子结构

## 2. LDPC码基础概念

LDPC码是一种线性分组码，其特点是**校验矩阵是稀疏的**（即矩阵中非零元素的密度很低）。

### 2.1 校验矩阵H与生成矩阵G

LDPC码由(n, k)参数定义：
- **n**：码字长度（编码后比特数）
- **k**：信息位长度（编码前比特数）
- **m = n - k**：校验位长度

校验矩阵H是一个m×n的稀疏矩阵，满足：
$$H \cdot G^T = 0$$

编码过程：
$$c = u \cdot G$$

其中u是k位信息向量，c是n位码字向量。

### 2.2 Tanner图表示

Tanner图是LDPC码的二分图表示，包含两类节点：
- **变量节点（Variable Nodes）**：对应码字比特，共n个
- **校验节点（Check Nodes）**：对应校验方程，共m个

当H(i,j)=1时，第j个变量节点与第i个校验节点相连。

### 2.3 代码演示：创建LDPC码示例

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
import networkx as nx
%matplotlib inline

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---------------------------------------------------
# 示例：创建一个小规模LDPC码
# 校验矩阵 H (4x8) - 对应 (8, 4) LDPC码
# ---------------------------------------------------

# 定义稀疏校验矩阵 H (4行8列)
H = np.array([
    [1, 1, 1, 1, 0, 0, 0, 0],
    [1, 0, 0, 0, 1, 1, 1, 0],
    [0, 1, 0, 0, 1, 0, 0, 1],
    [0, 0, 1, 0, 0, 1, 0, 1]
], dtype=np.int32)

print("校验矩阵 H (4x8):")
print(H)
print(f"\n矩阵维度: {H.shape[0]} x {H.shape[1]}")
print(f"码率: 1 - {H.shape[0]}/{H.shape[1]} = {1 - H.shape[0]/H.shape[1]:.2f}")
print(f"非零元素密度: {np.sum(H)/(H.shape[0]*H.shape[1]):.2%}")

In [ ]:
# ---------------------------------------------------
# 可视化Tanner图
# ---------------------------------------------------

def plot_tanner_graph(H):
    """绘制LDPC码的Tanner图"""
    m, n = H.shape  # m=校验节点数, n=变量节点数
    G = nx.Graph()
    
    # 添加变量节点 (v0 到 v{n-1})
    var_nodes = [f'v{i}' for i in range(n)]
    G.add_nodes_from(var_nodes, bipartite=0, node_type='variable')
    
    # 添加校验节点 (c0 到 c{m-1})
    check_nodes = [f'c{i}' for i in range(m)]
    G.add_nodes_from(check_nodes, bipartite=1, node_type='check')
    
    # 添加边（根据H矩阵）
    for i in range(m):
        for j in range(n):
            if H[i, j] == 1:
                G.add_edge(f'c{i}', f'v{j}')
    
    # 使用二分布局
    pos = {}
    # 变量节点在下方
    for i, node in enumerate(var_nodes):
        pos[node] = (i * 1.5, 0)
    # 校验节点在上方
    for i, node in enumerate(check_nodes):
        pos[node] = (i * 1.5 + 0.75, 2)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    
    # 绘制变量节点（圆形）
    nx.draw_networkx_nodes(G, pos, nodelist=var_nodes,
                          node_color='lightblue', node_size=800,
                          ax=ax)
    
    # 绘制校验节点（方形）
    nx.draw_networkx_nodes(G, pos, nodelist=check_nodes,
                          node_color='lightcoral', node_size=800,
                          ax=ax, node_shape='s')
    
    # 绘制边
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='gray')
    
    # 绘制标签
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)
    
    ax.set_title('LDPC码 Tanner图 (变量节点: 圆形, 校验节点: 方形)', fontsize=14)
    ax.axis('off')
    
    return fig

fig = plot_tanner_graph(H)
plt.tight_layout()
plt.show()

print("\nTanner图说明:")
print("- 圆形节点代表码字比特（变量节点）")
print("- 方形节点代表校验方程（校验节点）")
print("- 连接线表示该比特参与该校验方程")

## 3. Gallius构造法

Gallius构造法是一种结构化构造LDPC校验矩阵的方法，通过扩展小矩阵来创建大矩阵。

### 3.1 基本原理

给定一个基矩阵（种子矩阵），通过以下方式扩展：
1. 将每个元素替换为P×P的置换矩阵（或零矩阵）
2. 置换矩阵由循环移位参数决定

这种方法构造的矩阵具有规则的稀疏结构，便于理论分析。

In [ ]:
# ---------------------------------------------------
# Gallius构造法实现
# ---------------------------------------------------

def gallius_construction(base_matrix, P):
    """
    Gallius构造法：用置换矩阵扩展基矩阵
    
    参数:
        base_matrix: 基矩阵（二进制矩阵）
        P: 置换矩阵大小
    
    返回:
        H: 扩展后的校验矩阵
    """
    base_rows, base_cols = base_matrix.shape
    
    # 初始化扩展后的矩阵
    H = np.zeros((base_rows * P, base_cols * P), dtype=np.int32)
    
    for i in range(base_rows):
        for j in range(base_cols):
            if base_matrix[i, j] == 1:
                # 创建循环置换矩阵
                perm_matrix = np.eye(P, dtype=np.int32)
                # 循环右移j位（j作为移位因子）
                perm_matrix = np.roll(perm_matrix, j, axis=1)
            else:
                # 零矩阵
                perm_matrix = np.zeros((P, P), dtype=np.int32)
            
            # 将置换矩阵放到对应位置
            H[i*P:(i+1)*P, j*P:(j+1)*P] = perm_matrix
    
    return H

# 定义基矩阵（4x4）
base = np.array([
    [1, 1, 1, 1],
    [1, 1, 1, 0],
    [1, 1, 0, 1],
    [1, 0, 1, 1]
], dtype=np.int32)

print("基矩阵 (4x4):")
print(base)

# 使用P=3进行扩展
P = 3
H_gallius = gallius_construction(base, P)

print(f"\nGallius扩展后矩阵 ({base.shape[0]*P}x{base.shape[1]*P}):")
print(f"非零元素密度: {np.sum(H_gallius)/(H_gallius.shape[0]*H_gallius.shape[1]):.2%}")
print(f"\n矩阵形状: {H_gallius.shape}")

In [ ]:
# 可视化Gallius构造结果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 基矩阵可视化
ax1 = axes[0]
ax1.imshow(base, cmap='binary', interpolation='nearest')
ax1.set_title(f'基矩阵 ({base.shape[0]}x{base.shape[1]})', fontsize=12)
ax1.set_xlabel('列索引')
ax1.set_ylabel('行索引')
for i in range(base.shape[0]):
    for j in range(base.shape[1]):
        ax1.text(j, i, str(base[i, j]), ha='center', va='center')

# 扩展矩阵可视化
ax2 = axes[1]
ax2.imshow(H_gallius, cmap='binary', interpolation='nearest')
ax2.set_title(f'Gallius扩展矩阵 (P={P})', fontsize=12)
ax2.set_xlabel('列索引')
ax2.set_ylabel('行索引')

plt.tight_layout()
plt.show()

print("\nGallius构造特点:")
print("- 每个'1'扩展为P×P的循环置换矩阵")
print("- 每个'0'扩展为P×P的零矩阵")
print("- 移位因子由基矩阵中的位置决定")
print("- 保持稀疏性：密度与基矩阵相同")

## 4. Sum-Product算法（和积译码）

Sum-Product算法是LDPC码的标准消息传递译码算法，基于Tanner图的置信传播。

### 4.1 算法原理

算法在Tanner图上迭代传递两类消息：
- **校验节点到变量节点的消息**：来自同一校验节点的多个变量节点消息的函数
- **变量节点到校验节点的消息**：结合信道信息和校验节点消息

### 4.2 数学表达

设变量节点v的初始对数似然比(LLR)为：
$$L_v = \ln \frac{P(v=0|y)}{P(v=1|y)}$$

校验节点到变量节点的消息：
$$q_{c \to v} = 2 \cdot \text{atanh}\left(\prod_{v' \in N(c) \setminus v} \tanh(\frac{r_{v' \to c}}{2})\right)$$

变量节点到校验节点的消息：
$$r_{v \to c} = L_v + \sum_{c' \in N(v) \setminus c} q_{c' \to v}$$

### 4.3 迭代译码流程
1. 初始化：计算每个比特的对数似然比(LLR)
2. 校验节点更新：计算校验节点到变量节点的消息
3. 变量节点更新：计算变量节点到校验节点的消息
4. 判决：计算比特的后验LLR，判决输出
5. 终止条件：校验方程满足或达到最大迭代次数

In [ ]:
# ---------------------------------------------------
# Sum-Product算法实现
# ---------------------------------------------------

def sum_product_decode(H, llr, max_iter=50, min_llr_diff=1e-6):
    """
    Sum-Product（和积）译码算法
    
    参数:
        H: 校验矩阵 (m x n)
        llr: 信道LLR值 (n,) - 对数似然比
        max_iter: 最大迭代次数
        min_llr_diff: 最小LLR变化量，用于收敛判断
    
    返回:
        decoded_bits: 译码后的比特序列
        converged: 是否收敛
        iterations: 迭代次数
    """
    m, n = H.shape
    
    # 初始化变量节点消息（初始消息即为信道LLR）
    # q[v] 表示变量节点v收到的消息
    q = np.copy(llr)
    
    # 存储历史用于收敛判断
    prev_llr = np.copy(llr)
    
    for iteration in range(max_iter):
        # --- 校验节点更新 ---
        # 对每个校验节点c，计算到变量节点v的消息
        r = np.zeros((m, n))
        
        for c in range(m):
            # 获取该校验节点连接的变量节点
            var_nodes = np.where(H[c, :] == 1)[0]
            dv = len(var_nodes)  # 变量节点度
            
            if dv < 2:
                continue
            
            for v_idx, v in enumerate(var_nodes):
                # 获取其他变量节点的消息（排除当前v）
                other_vars = [var_nodes[j] for j in range(dv) if j != v_idx]
                
                if len(other_vars) == 0:
                    continue
                
                # tanh域计算：prod tanh(q/2)
                tanh_product = 1.0
                for v_other in other_vars:
                    tanh_val = np.tanh(q[v_other] / 2)
                    tanh_product *= tanh_val
                
                # 防止数值问题
                tanh_product = np.clip(tanh_product, -0.999, 0.999)
                
                # 计算校验节点消息
                r[c, v] = 2 * np.arctanh(tanh_product)
        
        # --- 变量节点更新 ---
        # 对每个变量节点v，计算到校验节点c的消息
        new_q = np.zeros(n)
        
        for v in range(n):
            # 获取该变量节点连接的校验节点
            check_nodes = np.where(H[:, v] == 1)[0]
            dc = len(check_nodes)  # 校验节点度
            
            # 后验LLR = 信道LLR + 所有校验节点消息
            posterior_llr = llr[v]
            for c in check_nodes:
                posterior_llr += r[c, v]
            
            new_q[v] = posterior_llr
        
        q = new_q
        
        # --- 收敛判断 ---
        llr_diff = np.max(np.abs(q - prev_llr))
        prev_llr = np.copy(q)
        
        if llr_diff < min_llr_diff:
            break
    
    # --- 判决 ---
    # LLR > 0 -> 判决为0, LLR < 0 -> 判决为1
    decoded_bits = (q < 0).astype(np.int32)
    converged = np.array_equal(np.mod(np.dot(decoded_bits, H.T), 2), np.zeros(m))
    
    return decoded_bits, converged, iteration + 1

print("Sum-Product算法实现完成")

In [ ]:
# ---------------------------------------------------
# LDPC编码器实现
# ---------------------------------------------------

def ldpc_encode(H, info_bits):
    """
    LDPC编码
    
    使用高斯消元法从H矩阵推导生成矩阵G，然后进行编码
    
    参数:
        H: 校验矩阵 (m x n)
        info_bits: 信息比特序列 (k,)
    
    返回:
        codeword: 编码后的码字 (n,)
    """
    m, n = H.shape
    k = n - m
    
    # 将H矩阵扩展为系统形式 [A | B]
    # 其中A是m×k的部分，B是m×m的部分
    # 通过高斯消元使B变为方阵
    
    # 使用伪逆方法或高斯消元
    # 这里使用简化的方法：假设H已经是系统形式
    # 对于一般的H，需要进行列置换和行变换
    
    # 简化实现：使用H矩阵的右半部分作为生成矩阵
    # 对于(k, n) LDPC码，G是k×n的矩阵
    
    # 计算有效的生成矩阵
    # 通过H求G的简化方法
    
    H_aug = H.copy()
    
    # 尝试高斯消元
    from scipy.linalg import lu
    
    # 将H复制用于消元
    H_work = H_aug.astype(float)
    m, n = H_work.shape
    
    # 向前消元（行阶梯形式）
    row, col = 0, 0
    pivot_cols = []
    while row < m and col < n:
        # 找到当前列的主元
        pivot_row = None
        for r in range(row, m):
            if abs(H_work[r, col]) > 1e-9:
                pivot_row = r
                break
        
        if pivot_row is None:
            col += 1
            continue
        
        # 交换行
        H_work[[row, pivot_row]] = H_work[[pivot_row, row]]
        
        # 归一化
        H_work[row] /= H_work[row, col]
        
        # 消去其他行
        for r in range(m):
            if r != row and abs(H_work[r, col]) > 1e-9:
                H_work[r] -= H_work[r, col] * H_work[row]
        
        pivot_cols.append(col)
        row += 1
        col += 1
    
    # 构建生成矩阵G
    # 找到非主元列对应的信息位
    info_cols = [c for c in range(n) if c not in pivot_cols]
    
    if len(info_cols) != k:
        # 如果不匹配，使用简化方法
        # 假设信息位对应前k位
        info_cols = list(range(k))
    
    # G矩阵：信息位列为单位阵，校验位列为消元结果
    G = np.zeros((k, n), dtype=np.float64)
    for i, ic in enumerate(info_cols):
        G[i, ic] = 1.0
    
    # 设置校验位部分
    for i, pc in enumerate(pivot_cols):
        if i < k:
            # 找到对应的信息位列
            # 校验位与信息位的关系
            for j, ic in enumerate(info_cols):
                G[j, pc] = -H_work[i, ic]
    
    G = np mod(np.round(G), 2).astype(np.int32)
    
    # 编码
    codeword = np.mod(np.dot(info_bits, G), 2)
    
    return codeword

print("LDPC编码器实现完成")

In [ ]:
# ---------------------------------------------------
# 端到端LDPC编码译码演示
# ---------------------------------------------------

np.random.seed(42)

# 创建简单的(8, 4) LDPC码
H_simple = np.array([
    [1, 1, 1, 1, 0, 0, 0, 0],
    [1, 0, 0, 0, 1, 1, 1, 0],
    [0, 1, 0, 0, 1, 0, 0, 1],
    [0, 0, 1, 0, 0, 1, 0, 1]
], dtype=np.int32)

m, n = H_simple.shape
k = n - m  # 信息位长度

print(f"LDPC码参数: ({n}, {k})")
print(f"码率: {k/n:.2f}")

# 生成随机信息位
info_bits = np.random.randint(0, 2, k)
print(f"\n信息比特: {info_bits}")

# 编码
codeword = ldpc_encode(H_simple, info_bits)
print(f"编码后码字: {codeword}")

# 验证校验关系
syndrome = np.mod(np.dot(codeword, H_simple.T), 2)
print(f"校验方程结果: {syndrome} (应为全0)")

# 添加噪声模拟BPSK调制和信道传输
# 0 -> +1, 1 -> -1
transmitted = 1 - 2 * codeword.astype(float)

# 信噪比Eb/N0 = 3dB
Eb_N0_db = 3.0
Eb_N0 = 10 ** (Eb_N0_db / 10)
noise_variance = 1 / (2 * (k/n) * Eb_N0)

# 添加高斯噪声
received = transmitted + np.sqrt(noise_variance) * np.random.randn(n)

# 计算LLR（对数似然比）
# 对于BPSK: LLR = 2*y/(sigma^2)
llr = 2 * received / noise_variance

print(f"\n接收信号LLR: {llr}")

# 译码
decoded, converged, iterations = sum_product_decode(H_simple, llr, max_iter=20)

print(f"\n译码结果: {decoded}")
print(f"译码成功: {np.array_equal(decoded, codeword)}")
print(f"收敛: {converged}, 迭代次数: {iterations}")

In [ ]:
# ---------------------------------------------------
# 误码率性能曲线
# ---------------------------------------------------

def simulate_ber(H, Eb_N0_db_range, n_trials=100, max_iter=50):
    """
    仿真LDPC码在不同信噪比下的误码率
    """
    m, n = H.shape
    k = n - m
    rate = k / n
    
    ber_results = []
    
    for Eb_N0_db in Eb_N0_db_range:
        Eb_N0 = 10 ** (Eb_N0_db / 10)
        noise_var = 1 / (2 * rate * Eb_N0)
        
        errors = 0
        total_bits = 0
        
        for trial in range(n_trials):
            # 生成随机信息位
            info_bits = np.random.randint(0, 2, k)
            
            # 编码
            codeword = ldpc_encode(H, info_bits)
            
            # BPSK调制
            transmitted = 1 - 2 * codeword.astype(float)
            
            # 信道
            received = transmitted + np.sqrt(noise_var) * np.random.randn(n)
            
            # LLR计算
            llr = 2 * received / noise_var
            
            # 译码
            decoded, _, _ = sum_product_decode(H, llr, max_iter=max_iter)
            
            # 统计错误
            errors += np.sum(decoded != codeword)
            total_bits += n
        
        ber = errors / total_bits
        ber_results.append(ber)
    
    return np.array(ber_results)

# 使用Gallius构造的较大LDPC码
base_matrix = np.array([
    [1, 1, 1, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 1],
    [0, 1, 1, 1]
], dtype=np.int32)
P = 5
H_large = gallius_construction(base_matrix, P)

print(f"LDPC码尺寸: {H_large.shape}")
print(f"码率: {1 - H_large.shape[0]/H_large.shape[1]:.2f}")

# 信噪比范围
Eb_N0_range = np.arange(1, 5, 0.5)

print("\n运行误码率仿真...")
ber_curve = simulate_ber(H_large, Eb_N0_range, n_trials=50, max_iter=30)

# 绘制BER曲线
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(Eb_N0_range, ber_curve, 'b-o', linewidth=2, markersize=6)
ax.set_xlabel('Eb/N0 (dB)', fontsize=12)
ax.set_ylabel('误码率 (BER)', fontsize=12)
ax.set_title('LDPC码误码率性能曲线', fontsize=14)
ax.grid(True, which='both', linestyle='--', alpha=0.7)
ax.set_xlim([Eb_N0_range[0], Eb_N0_range[-1]])
ax.set_ylim([1e-5, 1])
plt.tight_layout()
plt.show()

print("\n仿真完成")

## 5. 密度演化分析

密度演化（Density Evolution）是分析LDPC译码器性能的理论工具，用于跟踪消息概率分布的变化。

### 5.1 基本思想

在译码过程中，消息的概率密度函数会发生变化。密度演化追踪这个变化，预测译码是否收敛。

### 5.2 高斯近似

在加性高斯白噪声（AWGN）信道下，消息的密度可以用高斯分布近似：
$$f_X \sim \mathcal{N}(m, 2m)$$

其中均值m完全描述了分布特性。

### 5.3 密度演化递归

设变量节点消息的均值为$m_v$，校验节点消息的均值为$m_c$，则：
$$m_c = \phi^{-1}\left(1 - (1 - \phi(m_v)^{d_c-1})\right)$$
$$m_v = m_{ch} + (d_v - 1) m_c$$

其中$\phi(x)$是高斯密度演化中的映射函数。

In [ ]:
# ---------------------------------------------------
# 密度演化分析（高斯近似）
# ---------------------------------------------------

def phi_func(x):
    """
    密度演化中的phi函数
    使用近似公式提高数值稳定性
    """
    if isinstance(x, np.ndarray):
        result = np.copy(x)
        small_idx = x < 1e-10
        large_idx = x > 10
        medium_mask = ~small_idx & ~large_idx
        
        result[small_idx] = 1  # x -> 0 时 phi(x) -> 1
        result[large_idx] = 0  # x -> inf 时 phi(x) -> 0
        
        # 中等大小使用精确公式
        x_med = x[medium_mask]
        result[medium_mask] = np.sqrt(2 / np.pi) * np.exp(-x_med / 2) / np.sqrt(x_med) * (1 - np.exp(-x_med / 2))
        
        return result
    else:
        if x < 1e-10:
            return 1
        elif x > 10:
            return 0
        else:
            return np.sqrt(2 / np.pi) * np.exp(-x / 2) / np.sqrt(x) * (1 - np.exp(-x / 2))

def density_evolution(dv, dc, m_ch, max_iter=100, threshold=1e-6):
    """
    密度演化分析（高斯近似）
    
    参数:
        dv: 变量节点度
        dc: 校验节点度
        m_ch: 信道消息均值
        max_iter: 最大迭代次数
        threshold: 收敛阈值
    
    返回:
        m_v_history: 变量节点消息均值历史
        threshold_ebn0: 近似阈值Eb/N0
    """
    m_v = m_ch  # 初始化变量节点消息均值
    m_v_history = [m_v]
    
    for iteration in range(max_iter):
        # 校验节点更新
        phi_mv = phi_func(m_v)
        if phi_mv < threshold:
            m_c = 0
        else:
            m_c = phi_func(1 - (1 - phi_mv) ** (dc - 1))
        
        # 变量节点更新
        m_v_new = m_ch + (dv - 1) * m_c
        m_v_history.append(m_v_new)
        
        # 收敛判断
        if abs(m_v_new - m_v) < threshold:
            m_v = m_v_new
            break
        
        m_v = m_v_new
    
    return m_v_history

# 不同度分布的密度演化分析
print("密度演化分析 - 不同度分布比较")
print("=" * 50)

# 配置1: (3, 6)规则LDPC码
dv1, dc1 = 3, 6

# 配置2: (4, 8)规则LDPC码
dv2, dc2 = 4, 8

# 配置3: .irregular LDPC码近似
dv3, dc3 = 5, 6

# Eb/N0 到信道均值的转换
def ebn0_to_mch(ebn0_db, rate):
    """Eb/N0(dB)转换为信道消息均值（高斯近似）"""
    ebn0 = 10 ** (ebn0_db / 10)
    # 信道方差
    noise_var = 1 / (2 * rate * ebn0)
    # 信道LLR均值
    m_ch = 2 / noise_var
    return m_ch

# Eb/N0范围
ebn0_range = np.linspace(0.5, 4, 20)
rate = 0.5  # 假设码率

# 计算不同配置下的收敛性
results = {}
for name, dv, dc in [('(3,6)规则码', dv1, dc1), ('(4,8)规则码', dv2, dc2), ('(5,6)码', dv3, dc3)]:
    final_means = []
    for ebn0 in ebn0_range:
        m_ch = ebn0_to_mch(ebn0, rate)
        history = density_evolution(dv, dc, m_ch, max_iter=100)
        final_means.append(history[-1])
    results[name] = final_means

# 绘制结果
fig, ax = plt.subplots(figsize=(10, 6))
for name, means in results.items():
    ax.plot(ebn0_range, means, linewidth=2, label=name)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='收敛阈值')
ax.set_xlabel('Eb/N0 (dB)', fontsize=12)
ax.set_ylabel('变量节点消息均值', fontsize=12)
ax.set_title('密度演化分析 - 消息均值随信噪比变化', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.7)
plt.tight_layout()
plt.show()

print("\n密度演化分析说明:")
print("- 消息均值越大，表示译码越收敛")
print("- 阈值附近（均值接近0）表示译码临界成功/失败")
print("- 可以估计给定度分布的阈值信噪比")

## 6. 5G NR LDPC码结构

5G NR（New Radio）标准采用了LDPC码作为数据信道的编码方案。

### 6.1 基矩阵（Base Graph, BG）

5G NR定义了两种基矩阵：
- **BG1**: 用于小尺寸高码率场景（46x68）
- **BG2**: 用于大尺寸低码率场景（42x68）

移位因子P根据码块大小选择，使得扩展后的矩阵覆盖所需的码长。

### 6.2 5G NR LDPC特点

| 特性 | 描述 |
|------|------|
| 基矩阵大小 | BG1: 46x68, BG2: 42x68 |
| 码长范围 | 528 ~ 8448 |
| 码率范围 | 1/3 ~ 8/9 |
| 移位因子 | P ∈ {2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31} |

### 6.3 5G NR LDPC编码流程
1. 选择基矩阵（BG1或BG2）
2. 选择移位因子P
3. 根据P扩展基矩阵得到H
4. 对信息位进行编码

In [ ]:
# ---------------------------------------------------
# 5G NR LDPC基矩阵简化版
# ---------------------------------------------------

# 5G NR BG1的简化示例（实际标准使用完整的68x46矩阵）
# 这里展示简化版本以说明结构

def create_5g_nr_bg1():
    """
    创建5G NR BG1基矩阵的简化版本
    实际5G标准使用完整的基矩阵
    """
    # 简化的BG1结构：10x16
    # 实际标准是46x68
    BG1 = np.array([
        [1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
        [1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0]
    ], dtype=np.int32)
    return BG1

def create_5g_nr_bg2():
    """
    创建5G NR BG2基矩阵的简化版本
    实际5G标准使用完整的基矩阵
    """
    # 简化的BG2结构：8x16
    BG2 = np.array([
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
        [1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0],
        [0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0],
        [0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
    ], dtype=np.int32)
    return BG2

# 获取基矩阵
BG1 = create_5g_nr_bg1()
BG2 = create_5g_nr_bg2()

print("5G NR BG1 (简化版) 形状:", BG1.shape)
print("5G NR BG2 (简化版) 形状:", BG2.shape)

# 可视化基矩阵
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.imshow(BG1, cmap='RdBu', interpolation='nearest', aspect='auto')
ax1.set_title('BG1 (10x16) - 高码率场景', fontsize=12)
ax1.set_xlabel('列索引')
ax1.set_ylabel('行索引')

ax2 = axes[1]
ax2.imshow(BG2, cmap='RdBu', interpolation='nearest', aspect='auto')
ax2.set_title('BG2 (8x16) - 低码率场景', fontsize=12)
ax2.set_xlabel('列索引')
ax2.set_ylabel('行索引')

plt.tight_layout()
plt.show()

print("\n5G NR LDPC码特点:")
print("- BG1用于高码率（1/3以下）")
print("- BG2用于低码率（1/3以下）")
print("- 使用循环置换矩阵扩展")
print("- 移位因子根据码块大小选择")

In [ ]:
# ---------------------------------------------------
# 5G NR LDPC码仿真
# ---------------------------------------------------

# 使用Gallius构造模拟5G NR LDPC码
BG1 = create_5g_nr_bg1()
P = 7  # 移位因子

H_5g = gallius_construction(BG1, P)

print(f"5G NR LDPC码尺寸: {H_5g.shape}")
print(f"码率: {1 - H_5g.shape[0]/H_5g.shape[1]:.2f}")
print(f"非零元素密度: {np.sum(H_5g)/(H_5g.shape[0]*H_5g.shape[1]):.2%}")

# 仿真不同信噪比下的性能
Eb_N0_range = np.arange(1, 5, 0.5)
print("\n运行5G NR LDPC性能仿真...")
ber_5g = simulate_ber(H_5g, Eb_N0_range, n_trials=50, max_iter=30)

# 绘制BER曲线
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(Eb_N0_range, ber_5g, 'r-o', linewidth=2, markersize=6, label='5G NR LDPC')
ax.set_xlabel('Eb/N0 (dB)', fontsize=12)
ax.set_ylabel('误码率 (BER)', fontsize=12)
ax.set_title('5G NR LDPC码误码率性能', fontsize=14)
ax.legend()
ax.grid(True, which='both', linestyle='--', alpha=0.7)
ax.set_ylim([1e-5, 1])
plt.tight_layout()
plt.show()

print("\n5G NR LDPC标准要点:")
print("1. 使用两种基矩阵（BG1和BG2）适配不同码率")
print("2. 通过移位因子扩展支持多种码长")
print("3. 支持增量冗余混合自动重传请求（IR-HARQ）")
print("4. 在AWGN信道下性能接近香农极限")

## 7. 总结

本notebook介绍了LDPC码的核心内容：

| 主题 | 关键点 |
|------|--------|
| **基础概念** | 校验矩阵H、生成矩阵G、Tanner图 |
| **Gallius构造** | 用置换矩阵扩展基矩阵 |
| **Sum-Product算法** | 基于置信传播的消息传递译码 |
| **密度演化** | 分析译码收敛性的理论工具 |
| **5G NR LDPC** | BG基矩阵+移位因子结构 |

### LDPC码的优势
- **高性能**：接近香农极限的纠错能力
- **可扩展性好**：适用于从短码到长码的各种场景
- **并行译码**：消息传递可并行化，适合硬件实现
- **5G标准**：已成为5G NR数据信道的编码方案

## 8. 思考题

### 思考题 1
对于一个(10, 5) LDPC码，校验矩阵H是5x10的稀疏矩阵。请说明：
- 该码的码率是多少？
- Tanner图中有多少变量节点和校验节点？
- 如果某个变量节点度数为3，这意味着什么？

### 思考题 2
Sum-Product算法中，校验节点消息计算使用的公式为：
$$q_{c \to v} = 2 \cdot \text{atanh}\left(\prod_{v' \in N(c) \setminus v} \tanh(\frac{r_{v' \to c}}{2})\right)$$

请解释为什么这个公式使用tanh和atanh函数，而不是直接使用概率值的乘积。

### 思考题 3
在密度演化分析中，为什么使用高斯近似可以简化计算？请说明高斯近似的适用条件。

### 思考题 4
5G NR标准使用两种基矩阵（BG1和BG2），请分析：
- 什么情况下选择BG1，什么情况下选择BG2？
- 移位因子P的选择对码性能有什么影响？

### 思考题 5
假设使用Gallius构造法生成一个大尺寸LDPC码：
- 基矩阵大小为20x40
- 移位因子P=10

请计算最终校验矩阵的尺寸和非零元素密度。

### 思考题 6（扩展）
比较LDPC码与卷积码、Turbo码的优缺点。为什么5G NR选择了LDPC而不是Turbo码作为数据信道的编码方案？

---

**参考资源**
- R. Gallager, "Low-Density Parity-Check Codes" (1962) - 原始LDPC论文
- D. J. C. MacKay, "Good Error-Correcting Codes based on Very Sparse Matrices" (1999)
- 3GPP TS 38.212 - 5G NR复用和编码标准